# 04 - Gold Layer: Load Dimensional Model
Read from Silver zone (cleansed) and load the star schema (Gold zone).

**Medallion Architecture - Gold**: Business-level dimensional model (dims & facts).

In [0]:
%run ./00_setup_config

# 00 - Setup & Configuration
This notebook sets up the database, defines paths, and installs required libraries for the Food Inspection project.

All file paths are parameterized using widgets — no hardcoded values.

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 838.5/838.5 kB 36.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 775.6/775.6 kB 29.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 96.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 605.5/605.5 kB 22.1 MB/s eta 0:00:00
  Attempting uninstall: pyyaml
    Found existing installation: PyYAML 6.0
    Not uninstalling pyyaml at /databricks/python3/lib/python3.11/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-3740e9e9-5840-473a-9a32-3dfdfcfbd8e9
    Can't uninstall 'PyYAML'. No files were found to uninstall.
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.3
    Not uninstalling protobuf at /databricks/python3/lib/python3.11/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-3740e9e9-5840-473a-9a32-3dfdfcfbd8e9
    Can't uninstall 'protobuf'. No files were found to uninstall.
  Attempting uninstall: datab

## Widget Parameters

Volume Path      : /Volumes/workspace/food_inspection/raw_data
Raw Chicago Path : /Volumes/workspace/food_inspection/raw_data/Food_Inspections_20260411.csv
Raw Dallas Path  : /Volumes/workspace/food_inspection/raw_data/Restaurant_and_Food_Establishment_Inspections_(October_2016_to_January_2024)_20260411.csv
Database Name    : food_inspection


## Create Database

Database 'food_inspection' is ready.


## Verify Raw Data Files Exist

Raw data files in Volume:
  Food_Inspections_20260411.csv (330.70 MB)
  Restaurant_and_Food_Establishment_Inspections_(October_2016_to_January_2024)_20260411.csv (192.79 MB)


Configuration ready. All path variables are available via %run.


In [0]:
spark.sql(f"USE {DATABASE_NAME}")

DataFrame[]

In [0]:
from pyspark.sql.functions import (
    col, lit, when, coalesce, trim, upper, concat, concat_ws, expr,
    monotonically_increasing_id, row_number, dense_rank,
    year, month, dayofmonth, dayofweek, quarter, date_format,
    min as spark_min, max as spark_max, current_date, current_timestamp
)
from pyspark.sql.window import Window
from pyspark.sql.types import IntegerType
from delta.tables import DeltaTable

In [0]:
def get_etl_job_id():
    """Get the current notebook path, or 'interactive' if unavailable."""
    try:
        return dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
    except Exception:
        return "interactive"

ETL_JOB_ID = get_etl_job_id()

def add_gold_lineage(df):
    """Add Gold layer lineage columns: created_at, updated_at, etl_job_id."""
    return (
        df
        .withColumn("created_at", current_timestamp())
        .withColumn("updated_at", current_timestamp())
        .withColumn("etl_job_id", lit(ETL_JOB_ID))
    )

## 1. Load Silver Tables

In [0]:
df_chi_insp = spark.table(f"{DATABASE_NAME}.silver_chicago_inspections")
df_dal_insp = spark.table(f"{DATABASE_NAME}.silver_dallas_inspections")
df_chi_viol = spark.table(f"{DATABASE_NAME}.silver_chicago_violations")
df_dal_viol = spark.table(f"{DATABASE_NAME}.silver_dallas_violations")

print(f"Chicago inspections: {df_chi_insp.count()}")
print(f"Dallas inspections:  {df_dal_insp.count()}")
print(f"Chicago violations:  {df_chi_viol.count()}")
print(f"Dallas violations:   {df_dal_viol.count()}")

Chicago inspections: 221877
Dallas inspections:  54090
Chicago violations:  939571
Dallas violations:   308579


---
## 2. dim_date

In [0]:
# Get date range from both datasets
chi_dates = df_chi_insp.select(
    spark_min("Inspection_Date").alias("min_date"),
    spark_max("Inspection_Date").alias("max_date")
).collect()[0]

dal_dates = df_dal_insp.select(
    spark_min("Inspection_Date").alias("min_date"),
    spark_max("Inspection_Date").alias("max_date")
).collect()[0]

min_date = min(chi_dates["min_date"], dal_dates["min_date"])
max_date = max(chi_dates["max_date"], dal_dates["max_date"])

print(f"Date range: {min_date} to {max_date}")

Date range: 2010-01-04 to 2026-04-08


In [0]:
# Generate date dimension
df_dim_date = (
    spark.sql(f"""
        SELECT explode(sequence(
            to_date('{min_date}'),
            to_date('{max_date}'),
            interval 1 day
        )) AS full_date
    """)
    .withColumn("date_key", (year(col("full_date")) * 10000 + month(col("full_date")) * 100 + dayofmonth(col("full_date"))).cast(IntegerType()))
    .withColumn("day", dayofmonth(col("full_date")))
    .withColumn("month", month(col("full_date")))
    .withColumn("month_name", date_format(col("full_date"), "MMMM"))
    .withColumn("quarter", quarter(col("full_date")))
    .withColumn("year", year(col("full_date")))
    .withColumn("day_of_week", dayofweek(col("full_date")))
    .withColumn("day_name", date_format(col("full_date"), "EEEE"))
)

df_dim_date = add_gold_lineage(df_dim_date)

(
    df_dim_date.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(f"{DATABASE_NAME}.dim_date")
)

print(f"dim_date: {df_dim_date.count()} rows")
display(df_dim_date.limit(10))

dim_date: 5939 rows


full_date,date_key,day,month,month_name,quarter,year,day_of_week,day_name,created_at,updated_at,etl_job_id
2010-01-04,20100104,4,1,January,1,2010,2,Monday,2026-04-12T18:48:11.623Z,2026-04-12T18:48:11.623Z,/Repos/ravi.ara@northeastern.edu/FoodInspection_Chicago_Dallas/notebooks/04_gold_dim_load
2010-01-05,20100105,5,1,January,1,2010,3,Tuesday,2026-04-12T18:48:11.623Z,2026-04-12T18:48:11.623Z,/Repos/ravi.ara@northeastern.edu/FoodInspection_Chicago_Dallas/notebooks/04_gold_dim_load
2010-01-06,20100106,6,1,January,1,2010,4,Wednesday,2026-04-12T18:48:11.623Z,2026-04-12T18:48:11.623Z,/Repos/ravi.ara@northeastern.edu/FoodInspection_Chicago_Dallas/notebooks/04_gold_dim_load
2010-01-07,20100107,7,1,January,1,2010,5,Thursday,2026-04-12T18:48:11.623Z,2026-04-12T18:48:11.623Z,/Repos/ravi.ara@northeastern.edu/FoodInspection_Chicago_Dallas/notebooks/04_gold_dim_load
2010-01-08,20100108,8,1,January,1,2010,6,Friday,2026-04-12T18:48:11.623Z,2026-04-12T18:48:11.623Z,/Repos/ravi.ara@northeastern.edu/FoodInspection_Chicago_Dallas/notebooks/04_gold_dim_load
2010-01-09,20100109,9,1,January,1,2010,7,Saturday,2026-04-12T18:48:11.623Z,2026-04-12T18:48:11.623Z,/Repos/ravi.ara@northeastern.edu/FoodInspection_Chicago_Dallas/notebooks/04_gold_dim_load
2010-01-10,20100110,10,1,January,1,2010,1,Sunday,2026-04-12T18:48:11.623Z,2026-04-12T18:48:11.623Z,/Repos/ravi.ara@northeastern.edu/FoodInspection_Chicago_Dallas/notebooks/04_gold_dim_load
2010-01-11,20100111,11,1,January,1,2010,2,Monday,2026-04-12T18:48:11.623Z,2026-04-12T18:48:11.623Z,/Repos/ravi.ara@northeastern.edu/FoodInspection_Chicago_Dallas/notebooks/04_gold_dim_load
2010-01-12,20100112,12,1,January,1,2010,3,Tuesday,2026-04-12T18:48:11.623Z,2026-04-12T18:48:11.623Z,/Repos/ravi.ara@northeastern.edu/FoodInspection_Chicago_Dallas/notebooks/04_gold_dim_load
2010-01-13,20100113,13,1,January,1,2010,4,Wednesday,2026-04-12T18:48:11.623Z,2026-04-12T18:48:11.623Z,/Repos/ravi.ara@northeastern.edu/FoodInspection_Chicago_Dallas/notebooks/04_gold_dim_load


---
## 3. dim_restaurant

### 3.1 Build staging data (deduplicated per business key)
Business key: `restaurant_name + source_city + license_number`

Tracked SCD2 attributes: `facility_type`, `risk_category`, `aka_name`

In [0]:
# Chicago restaurants - deduplicate by business key, keep latest values
# Add Inspection_ID as tiebreaker for rows with same date
w_chi = Window.partitionBy("DBA_Name", "source_city", "License").orderBy(col("Inspection_Date").desc(), col("Inspection_ID").desc())
df_chi_restaurants = (
    df_chi_insp
    .withColumn("_rn", row_number().over(w_chi))
    .filter(col("_rn") == 1)
    .select(
        col("DBA_Name").alias("restaurant_name"),
        col("AKA_Name").alias("aka_name"),
        col("License").cast("string").alias("license_number"),
        col("Facility_Type").alias("facility_type"),
        col("Risk").alias("risk_category"),
        col("source_city")
    )
    .dropDuplicates(["restaurant_name", "source_city", "license_number"])  # safety net
)

# Dallas restaurants - deduplicate by business key
w_dal = Window.partitionBy("Restaurant_Name", "source_city").orderBy(col("Inspection_Date").desc(), col("Inspection_ID").desc())
df_dal_restaurants = (
    df_dal_insp
    .withColumn("_rn", row_number().over(w_dal))
    .filter(col("_rn") == 1)
    .select(
        col("Restaurant_Name").alias("restaurant_name"),
        lit(None).cast("string").alias("aka_name"),
        lit(None).cast("string").alias("license_number"),
        lit(None).cast("string").alias("facility_type"),
        lit(None).cast("string").alias("risk_category"),
        col("source_city")
    )
    .dropDuplicates(["restaurant_name", "source_city", "license_number"])  # safety net
)

# Union staging data
df_staging_restaurant = df_chi_restaurants.unionByName(df_dal_restaurants)

df_staging_restaurant.createOrReplaceTempView("staging_restaurant")
print(f"Staging restaurants: {df_staging_restaurant.count()} (Chicago: {df_chi_restaurants.count()}, Dallas: {df_dal_restaurants.count()})")

Staging restaurants: 45544 (Chicago: 36857, Dallas: 8687)


### 3.2 SCD Type 2 Merge on dim_restaurant
- First run: creates the table with initial load
- Subsequent runs: expires changed rows, inserts new current rows

In [0]:
table_name = f"{DATABASE_NAME}.dim_restaurant"

# Check if dim_restaurant exists as a Delta table
table_exists = False
try:
    if spark.catalog.tableExists(table_name):
        # Verify it's actually a Delta table
        table_format = spark.sql(f"DESCRIBE DETAIL {table_name}").select("format").collect()[0][0]
        if table_format == "delta":
            target = DeltaTable.forName(spark, table_name)
            table_exists = True
            print("dim_restaurant exists as Delta — applying SCD Type 2 merge...")
        else:
            print(f"dim_restaurant exists but is {table_format} format — dropping and recreating as Delta...")
            spark.sql(f"DROP TABLE IF EXISTS {table_name}")
except Exception as e:
    print(f"Table check: {e} — will create fresh.")
    table_exists = False

if table_exists:

    # Step 1: Expire changed rows
    target.alias("t").merge(
        df_staging_restaurant.alias("s"),
        """
        t.restaurant_name = s.restaurant_name
        AND t.source_city = s.source_city
        AND COALESCE(t.license_number, '') = COALESCE(s.license_number, '')
        AND t.is_current = True
        """
    ).whenMatchedUpdate(
        condition="""
            COALESCE(t.facility_type, '') != COALESCE(s.facility_type, '')
            OR COALESCE(t.risk_category, '') != COALESCE(s.risk_category, '')
            OR COALESCE(t.aka_name, '') != COALESCE(s.aka_name, '')
        """,
        set={
            "effective_end_date": current_date(),
            "is_current": lit(False),
            "updated_at": current_timestamp()
        }
    ).execute()

    print("Step 1: Expired changed rows.")

    # Step 2: Insert new current rows for changed + brand new records
    new_or_changed = spark.sql(f"""
        SELECT s.*
        FROM staging_restaurant s
        LEFT JOIN {table_name} t
        ON s.restaurant_name = t.restaurant_name
           AND s.source_city = t.source_city
           AND COALESCE(s.license_number, '') = COALESCE(t.license_number, '')
           AND t.is_current = True
        WHERE t.restaurant_key IS NULL
    """)

    if new_or_changed.count() > 0:
        max_key = spark.sql(f"SELECT COALESCE(MAX(restaurant_key), 0) AS mk FROM {table_name}").collect()[0]["mk"]

        new_rows = (
            new_or_changed
            .withColumn("restaurant_key", monotonically_increasing_id() + max_key + 1)
            .withColumn("effective_start_date", current_date())
            .withColumn("effective_end_date", lit("9999-12-31").cast("date"))
            .withColumn("is_current", lit(True))
        )
        new_rows = add_gold_lineage(new_rows)
        new_rows.write.format("delta").mode("append").saveAsTable(table_name)
        print(f"Step 2: Inserted {new_rows.count()} new/updated rows.")
    else:
        print("Step 2: No new or changed records.")

else:
    print("dim_restaurant does not exist — creating initial load with APPEND mode...")

    df_dim_restaurant = (
        df_staging_restaurant
        .withColumn("restaurant_key", monotonically_increasing_id())
        .withColumn("effective_start_date", current_date())
        .withColumn("effective_end_date", lit("9999-12-31").cast("date"))
        .withColumn("is_current", lit(True))
    )
    df_dim_restaurant = add_gold_lineage(df_dim_restaurant)

    # Use append mode — saveAsTable creates the Delta table if it doesn't exist
    # Never use overwrite for SCD2 tables to preserve history on re-runs
    (
        df_dim_restaurant.write
        .format("delta")
        .mode("append")
        .saveAsTable(table_name)
    )
    print(f"Initial load (append): {df_dim_restaurant.count()} rows")

dim_restaurant exists as Delta — applying SCD Type 2 merge...
Step 1: Expired changed rows.
Step 2: Inserted 0 new/updated rows.


In [0]:
# Show SCD2 state
display(spark.sql(f"""
    SELECT is_current, COUNT(*) AS row_count
    FROM {DATABASE_NAME}.dim_restaurant
    GROUP BY is_current
"""))

dim_restaurant = spark.table(f"{DATABASE_NAME}.dim_restaurant")
print(f"dim_restaurant total: {dim_restaurant.count()} rows")
display(dim_restaurant.filter(col("is_current") == True).limit(10))

is_current,row_count
true,45548
false,4


dim_restaurant total: 45552 rows


restaurant_name,aka_name,license_number,facility_type,risk_category,source_city,restaurant_key,effective_start_date,effective_end_date,is_current,created_at,updated_at,etl_job_id
123 CONCESSION SHOP,123 CONCESSION SHOP,1273598,Grocery Store,Risk 3 (Low),Chicago,0,2026-04-12,9999-12-31,true,2026-04-12T09:47:15.990Z,2026-04-12T09:47:15.990Z,/Repos/.internal/c2113c435d_commits/c7ef81b083cbfbca18d69c4a9f9453646f67d0de/notebooks/04_gold_dim_load
1ST CHOP SUEY,1ST CHOP SUEY,58750,Restaurant,Risk 1 (High),Chicago,1,2026-04-12,9999-12-31,true,2026-04-12T09:47:15.990Z,2026-04-12T09:47:15.990Z,/Repos/.internal/c2113c435d_commits/c7ef81b083cbfbca18d69c4a9f9453646f67d0de/notebooks/04_gold_dim_load
1UP FOOD ARACADE,1UP FOOD ARACADE,2785939,Wholesale,Risk 2 (Medium),Chicago,2,2026-04-12,9999-12-31,true,2026-04-12T09:47:15.990Z,2026-04-12T09:47:15.990Z,/Repos/.internal/c2113c435d_commits/c7ef81b083cbfbca18d69c4a9f9453646f67d0de/notebooks/04_gold_dim_load
"1st HUNAN SPRING, INC",1ST HUNAN SPRING,2303648,Restaurant,Risk 1 (High),Chicago,3,2026-04-12,9999-12-31,true,2026-04-12T09:47:15.990Z,2026-04-12T09:47:15.990Z,/Repos/.internal/c2113c435d_commits/c7ef81b083cbfbca18d69c4a9f9453646f67d0de/notebooks/04_gold_dim_load
2 POTZ & A PAN EATERY,2 POTZ & A PAN EATERY,2626283,Restaurant,Risk 1 (High),Chicago,4,2026-04-12,9999-12-31,true,2026-04-12T09:47:15.990Z,2026-04-12T09:47:15.990Z,/Repos/.internal/c2113c435d_commits/c7ef81b083cbfbca18d69c4a9f9453646f67d0de/notebooks/04_gold_dim_load
2015 SEE THRU CHINESE KITCHEN,SEE THRU CHINESE KITCHEN,2390636,Restaurant,Risk 1 (High),Chicago,5,2026-04-12,9999-12-31,true,2026-04-12T09:47:15.990Z,2026-04-12T09:47:15.990Z,/Repos/.internal/c2113c435d_commits/c7ef81b083cbfbca18d69c4a9f9453646f67d0de/notebooks/04_gold_dim_load
3JJJ'S BETTER TASTE JAMAICAN JERK RESTAURANT,3JJJ'S BETTER TASTE JAMAICAN JERK RESTAURANT,2469229,Mobile Food Dispenser,Risk 3 (Low),Chicago,6,2026-04-12,9999-12-31,true,2026-04-12T09:47:15.990Z,2026-04-12T09:47:15.990Z,/Repos/.internal/c2113c435d_commits/c7ef81b083cbfbca18d69c4a9f9453646f67d0de/notebooks/04_gold_dim_load
"4 EVER YOUNG DAY CARE,INC.","4 EVER YOUNG DAY CARE,INC.",2084445,Daycare Combo 1586,Risk 1 (High),Chicago,7,2026-04-12,9999-12-31,true,2026-04-12T09:47:15.990Z,2026-04-12T09:47:15.990Z,/Repos/.internal/c2113c435d_commits/c7ef81b083cbfbca18d69c4a9f9453646f67d0de/notebooks/04_gold_dim_load
"4 YOLKS,INC",4 YOLKS,2307773,Restaurant,Risk 1 (High),Chicago,8,2026-04-12,9999-12-31,true,2026-04-12T09:47:15.990Z,2026-04-12T09:47:15.990Z,/Repos/.internal/c2113c435d_commits/c7ef81b083cbfbca18d69c4a9f9453646f67d0de/notebooks/04_gold_dim_load
5 BOROUGHS PIZZA & SUBS,5 BOROUGHS PIZZA,2269618,Restaurant,Risk 2 (Medium),Chicago,9,2026-04-12,9999-12-31,true,2026-04-12T09:47:15.990Z,2026-04-12T09:47:15.990Z,/Repos/.internal/c2113c435d_commits/c7ef81b083cbfbca18d69c4a9f9453646f67d0de/notebooks/04_gold_dim_load


---
## 4. dim_location

In [0]:
# Chicago locations (Latitude/Longitude already cleaned in Silver; re-cast as safety net)
df_chi_locations = (
    df_chi_insp.select(
        col("Address").alias("address"),
        col("City").alias("city"),
        col("State").alias("state"),
        col("Zip").alias("zip"),
        expr("try_cast(Latitude as double)").alias("latitude"),
        expr("try_cast(Longitude as double)").alias("longitude"),
        col("source_city")
    ).distinct()
)

# Dallas locations (Latitude/Longitude already cleaned in Silver; re-cast as safety net)
df_dal_locations = (
    df_dal_insp.select(
        col("Street_Address").alias("address"),
        col("City").alias("city"),
        col("State").alias("state"),
        col("Zip_Code").alias("zip"),
        expr("try_cast(Latitude as double)").alias("latitude"),
        expr("try_cast(Longitude as double)").alias("longitude"),
        col("source_city")
    ).distinct()
)

# Union and assign surrogate keys
df_dim_location = (
    df_chi_locations.unionByName(df_dal_locations)
    .distinct()
    .withColumn("location_key", monotonically_increasing_id())
)

df_dim_location = add_gold_lineage(df_dim_location)

(
    df_dim_location.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(f"{DATABASE_NAME}.dim_location")
)

print(f"dim_location: {df_dim_location.count()} rows")
display(df_dim_location.limit(10))

dim_location: 34148 rows


address,city,state,zip,latitude,longitude,source_city,location_key,created_at,updated_at,etl_job_id
3500 N CLARK ST,CHICAGO,IL,60657,41.94559291704,-87.65539147593,Chicago,0,2026-04-12T18:48:49.812Z,2026-04-12T18:48:49.812Z,/Repos/ravi.ara@northeastern.edu/FoodInspection_Chicago_Dallas/notebooks/04_gold_dim_load
1 W DIVISION ST,CHICAGO,IL,60610,41.90383104554,-87.62874004776,Chicago,1,2026-04-12T18:48:49.812Z,2026-04-12T18:48:49.812Z,/Repos/ravi.ara@northeastern.edu/FoodInspection_Chicago_Dallas/notebooks/04_gold_dim_load
6322 W GRAND AVE,CHICAGO,IL,60639,41.92226483352,-87.78383149142,Chicago,2,2026-04-12T18:48:49.812Z,2026-04-12T18:48:49.812Z,/Repos/ravi.ara@northeastern.edu/FoodInspection_Chicago_Dallas/notebooks/04_gold_dim_load
508 E PERSHING RD BLDG SUITE 1,CHICAGO,IL,60653,41.82398200091,-87.61410050374,Chicago,3,2026-04-12T18:48:49.812Z,2026-04-12T18:48:49.812Z,/Repos/ravi.ara@northeastern.edu/FoodInspection_Chicago_Dallas/notebooks/04_gold_dim_load
3322-3324 W 26TH ST,CHICAGO,IL,60623,41.84457881432,-87.7085399064,Chicago,4,2026-04-12T18:48:49.812Z,2026-04-12T18:48:49.812Z,/Repos/ravi.ara@northeastern.edu/FoodInspection_Chicago_Dallas/notebooks/04_gold_dim_load
2305 W DEVON AVE,CHICAGO,IL,60659,41.99765846224,-87.68767204214,Chicago,5,2026-04-12T18:48:49.812Z,2026-04-12T18:48:49.812Z,/Repos/ravi.ara@northeastern.edu/FoodInspection_Chicago_Dallas/notebooks/04_gold_dim_load
5153 W LAKE ST,CHICAGO,IL,60644,41.88706805237,-87.75500577148,Chicago,6,2026-04-12T18:48:49.812Z,2026-04-12T18:48:49.812Z,/Repos/ravi.ara@northeastern.edu/FoodInspection_Chicago_Dallas/notebooks/04_gold_dim_load
3024 N LARAMIE AVE,CHICAGO,IL,60641,41.93579920424,-87.75673008967,Chicago,7,2026-04-12T18:48:49.812Z,2026-04-12T18:48:49.812Z,/Repos/ravi.ara@northeastern.edu/FoodInspection_Chicago_Dallas/notebooks/04_gold_dim_load
4518-4528 S CICERO AVE,CHICAGO,IL,60638,41.81066961512,-87.74338370313,Chicago,8,2026-04-12T18:48:49.812Z,2026-04-12T18:48:49.812Z,/Repos/ravi.ara@northeastern.edu/FoodInspection_Chicago_Dallas/notebooks/04_gold_dim_load
5971 W MADISON ST,CHICAGO,IL,60644,41.87997806534,-87.77458167014,Chicago,9,2026-04-12T18:48:49.812Z,2026-04-12T18:48:49.812Z,/Repos/ravi.ara@northeastern.edu/FoodInspection_Chicago_Dallas/notebooks/04_gold_dim_load


---
## 5. dim_inspection_type

In [0]:
# Distinct inspection types from both cities
df_chi_types = df_chi_insp.select(col("Inspection_Type").alias("inspection_type_name"), col("source_city")).distinct()
df_dal_types = df_dal_insp.select(col("Inspection_Type").alias("inspection_type_name"), col("source_city")).distinct()

df_dim_inspection_type = (
    df_chi_types.unionByName(df_dal_types)
    .distinct()
    .withColumn("inspection_type_key", monotonically_increasing_id())
)

df_dim_inspection_type = add_gold_lineage(df_dim_inspection_type)

(
    df_dim_inspection_type.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(f"{DATABASE_NAME}.dim_inspection_type")
)

print(f"dim_inspection_type: {df_dim_inspection_type.count()} rows")
display(df_dim_inspection_type)

dim_inspection_type: 64 rows


inspection_type_name,source_city,inspection_type_key,created_at,updated_at,etl_job_id
Short Form Fire-Complaint,Chicago,0,2026-04-12T18:48:56.686Z,2026-04-12T18:48:56.686Z,/Repos/ravi.ara@northeastern.edu/FoodInspection_Chicago_Dallas/notebooks/04_gold_dim_load
LICENSE DAYCARE 1586,Chicago,1,2026-04-12T18:48:56.686Z,2026-04-12T18:48:56.686Z,/Repos/ravi.ara@northeastern.edu/FoodInspection_Chicago_Dallas/notebooks/04_gold_dim_load
License,Chicago,2,2026-04-12T18:48:56.686Z,2026-04-12T18:48:56.686Z,/Repos/ravi.ara@northeastern.edu/FoodInspection_Chicago_Dallas/notebooks/04_gold_dim_load
COVID COMPLAINT,Chicago,3,2026-04-12T18:48:56.686Z,2026-04-12T18:48:56.686Z,/Repos/ravi.ara@northeastern.edu/FoodInspection_Chicago_Dallas/notebooks/04_gold_dim_load
Tag Removal,Chicago,4,2026-04-12T18:48:56.686Z,2026-04-12T18:48:56.686Z,/Repos/ravi.ara@northeastern.edu/FoodInspection_Chicago_Dallas/notebooks/04_gold_dim_load
DAY CARE LICENSE RENEWAL,Chicago,5,2026-04-12T18:48:56.686Z,2026-04-12T18:48:56.686Z,/Repos/ravi.ara@northeastern.edu/FoodInspection_Chicago_Dallas/notebooks/04_gold_dim_load
REINSPECTION,Chicago,6,2026-04-12T18:48:56.686Z,2026-04-12T18:48:56.686Z,/Repos/ravi.ara@northeastern.edu/FoodInspection_Chicago_Dallas/notebooks/04_gold_dim_load
Special Events (Festivals),Chicago,7,2026-04-12T18:48:56.686Z,2026-04-12T18:48:56.686Z,/Repos/ravi.ara@northeastern.edu/FoodInspection_Chicago_Dallas/notebooks/04_gold_dim_load
LICENSE RENEWAL INSPECTION FOR DAYCARE,Chicago,8,2026-04-12T18:48:56.686Z,2026-04-12T18:48:56.686Z,/Repos/ravi.ara@northeastern.edu/FoodInspection_Chicago_Dallas/notebooks/04_gold_dim_load
Illegal Operation,Chicago,9,2026-04-12T18:48:56.686Z,2026-04-12T18:48:56.686Z,/Repos/ravi.ara@northeastern.edu/FoodInspection_Chicago_Dallas/notebooks/04_gold_dim_load


---
## 6. dim_violation
Stores both Chicago and Dallas violation codes (they don't need to match).

In [0]:
# Chicago distinct violations
df_chi_viol_dim = (
    df_chi_viol.select(
        col("violation_code"),
        col("violation_description"),
        lit(None).cast(IntegerType()).alias("violation_points"),
        col("source_city")
    ).distinct()
)

# Dallas distinct violations
df_dal_viol_dim = (
    df_dal_viol.select(
        col("violation_code"),
        col("violation_description"),
        col("violation_points"),
        col("source_city")
    ).distinct()
)

# Union and assign surrogate keys
df_dim_violation = (
    df_chi_viol_dim.unionByName(df_dal_viol_dim)
    .distinct()
    .withColumn("violation_key", monotonically_increasing_id())
)

df_dim_violation = add_gold_lineage(df_dim_violation)

(
    df_dim_violation.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(f"{DATABASE_NAME}.dim_violation")
)

print(f"dim_violation: {df_dim_violation.count()} rows")
display(df_dim_violation.limit(20))

dim_violation: 990 rows


violation_code,violation_description,violation_points,source_city,violation_key,created_at,updated_at,etl_job_id
27,FOOD ADDITIVES: APPROVED AND PROPERLY USED,null,Chicago,0,2026-04-12T18:49:03.285Z,2026-04-12T18:49:03.285Z,/Repos/ravi.ara@northeastern.edu/FoodInspection_Chicago_Dallas/notebooks/04_gold_dim_load
8,HANDS CLEAN & PROPERLY WASHED,null,Chicago,1,2026-04-12T18:49:03.285Z,2026-04-12T18:49:03.285Z,/Repos/ravi.ara@northeastern.edu/FoodInspection_Chicago_Dallas/notebooks/04_gold_dim_load
22,"DISH MACHINES: PROVIDED WITH ACCURATE THERMOMETERS, CHEMICAL TEST KITS AND SUITABLE GAUGE COCK",null,Chicago,2,2026-04-12T18:49:03.285Z,2026-04-12T18:49:03.285Z,/Repos/ravi.ara@northeastern.edu/FoodInspection_Chicago_Dallas/notebooks/04_gold_dim_load
2,FACILITIES TO MAINTAIN PROPER TEMPERATURE,null,Chicago,3,2026-04-12T18:49:03.285Z,2026-04-12T18:49:03.285Z,/Repos/ravi.ara@northeastern.edu/FoodInspection_Chicago_Dallas/notebooks/04_gold_dim_load
6,"PROPER EATING, TASTING, DRINKING, OR TOBACCO USE",null,Chicago,4,2026-04-12T18:49:03.285Z,2026-04-12T18:49:03.285Z,/Repos/ravi.ara@northeastern.edu/FoodInspection_Chicago_Dallas/notebooks/04_gold_dim_load
29,COMPLIANCE WITH VARIANCE/SPECIALIZED PROCESS/HACCP,null,Chicago,5,2026-04-12T18:49:03.285Z,2026-04-12T18:49:03.285Z,/Repos/ravi.ara@northeastern.edu/FoodInspection_Chicago_Dallas/notebooks/04_gold_dim_load
13,"NO EVIDENCE OF RODENT OR INSECT INFESTATION, NO BIRDS, TURTLES OR OTHER ANIMALS",null,Chicago,6,2026-04-12T18:49:03.285Z,2026-04-12T18:49:03.285Z,/Repos/ravi.ara@northeastern.edu/FoodInspection_Chicago_Dallas/notebooks/04_gold_dim_load
7,WASH AND RINSE WATER: CLEAN AND PROPER TEMPERATURE,null,Chicago,7,2026-04-12T18:49:03.285Z,2026-04-12T18:49:03.285Z,/Repos/ravi.ara@northeastern.edu/FoodInspection_Chicago_Dallas/notebooks/04_gold_dim_load
23,PROPER DATE MARKING AND DISPOSITION,null,Chicago,8,2026-04-12T18:49:03.285Z,2026-04-12T18:49:03.285Z,/Repos/ravi.ara@northeastern.edu/FoodInspection_Chicago_Dallas/notebooks/04_gold_dim_load
50,HOT & COLD WATER AVAILABLE; ADEQUATE PRESSURE,null,Chicago,9,2026-04-12T18:49:03.285Z,2026-04-12T18:49:03.285Z,/Repos/ravi.ara@northeastern.edu/FoodInspection_Chicago_Dallas/notebooks/04_gold_dim_load


---
## 7. fact_inspection

In [0]:
# Reload dims to get surrogate keys (filter dim_restaurant for current rows only)
dim_restaurant = spark.table(f"{DATABASE_NAME}.dim_restaurant").filter(col("is_current") == True)
dim_location = spark.table(f"{DATABASE_NAME}.dim_location")
dim_inspection_type = spark.table(f"{DATABASE_NAME}.dim_inspection_type")
dim_date = spark.table(f"{DATABASE_NAME}.dim_date")

### 7.1 Chicago Fact

In [0]:
df_chi_fact = (
    df_chi_insp
    .join(
        dim_restaurant.select("restaurant_key", "restaurant_name", "license_number", "source_city"),
        (df_chi_insp["DBA_Name"] == dim_restaurant["restaurant_name"]) &
        (df_chi_insp["License"].cast("string") == dim_restaurant["license_number"]) &
        (df_chi_insp["source_city"] == dim_restaurant["source_city"]),
        "left"
    )
    .join(
        dim_location,
        (df_chi_insp["Address"] == dim_location["address"]) &
        (df_chi_insp["Zip"] == dim_location["zip"]) &
        (df_chi_insp["source_city"] == dim_location["source_city"]),
        "left"
    )
    .join(
        dim_inspection_type,
        (df_chi_insp["Inspection_Type"] == dim_inspection_type["inspection_type_name"]) &
        (df_chi_insp["source_city"] == dim_inspection_type["source_city"]),
        "left"
    )
    .join(
        dim_date,
        df_chi_insp["Inspection_Date"] == dim_date["full_date"],
        "left"
    )
    .select(
        df_chi_insp["Inspection_ID"].alias("inspection_id"),
        col("restaurant_key"),
        col("location_key"),
        col("inspection_type_key"),
        col("date_key"),
        df_chi_insp["Results"].alias("inspection_result"),
        df_chi_insp["Inspection_Score"].alias("inspection_score"),
        df_chi_insp["source_city"]
    )
)

### 7.2 Dallas Fact

In [0]:
df_dal_fact = (
    df_dal_insp
    .join(
        dim_restaurant.filter(col("is_current") == True).select("restaurant_key", "restaurant_name", "source_city"),
        (df_dal_insp["Restaurant_Name"] == dim_restaurant["restaurant_name"]) &
        (df_dal_insp["source_city"] == dim_restaurant["source_city"]),
        "left"
    )
    .join(
        dim_location,
        (df_dal_insp["Street_Address"] == dim_location["address"]) &
        (df_dal_insp["Zip_Code"] == dim_location["zip"]) &
        (df_dal_insp["source_city"] == dim_location["source_city"]),
        "left"
    )
    .join(
        dim_inspection_type,
        (df_dal_insp["Inspection_Type"] == dim_inspection_type["inspection_type_name"]) &
        (df_dal_insp["source_city"] == dim_inspection_type["source_city"]),
        "left"
    )
    .join(
        dim_date,
        df_dal_insp["Inspection_Date"] == dim_date["full_date"],
        "left"
    )
    .select(
        df_dal_insp["Inspection_ID"].alias("inspection_id"),
        col("restaurant_key"),
        col("location_key"),
        col("inspection_type_key"),
        col("date_key"),
        # Derive inspection result from score for Dallas
        when(df_dal_insp["Inspection_Score"] >= 90, lit("Pass"))
        .when(df_dal_insp["Inspection_Score"] >= 80, lit("Pass w/ Conditions"))
        .when(df_dal_insp["Inspection_Score"] >= 70, lit("Fail"))
        .when(df_dal_insp["Inspection_Score"] < 70, lit("Fail"))
        .otherwise(lit(None).cast("string"))
        .alias("inspection_result"),
        df_dal_insp["Inspection_Score"].alias("inspection_score"),
        df_dal_insp["source_city"]
    )
)

### 7.3 Union & Write Fact Table

In [0]:
df_fact_inspection = (
    df_chi_fact.unionByName(df_dal_fact)
    .withColumn("inspection_key", monotonically_increasing_id())
)

df_fact_inspection = add_gold_lineage(df_fact_inspection)

(
    df_fact_inspection.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(f"{DATABASE_NAME}.fact_inspection")
)

print(f"fact_inspection: {df_fact_inspection.count()} rows")
display(df_fact_inspection.limit(10))

fact_inspection: 357094 rows


inspection_id,restaurant_key,location_key,inspection_type_key,date_key,inspection_result,inspection_score,source_city,inspection_key,created_at,updated_at,etl_job_id
2633665,7788,11455,2,20260408,Pass,90,Chicago,0,2026-04-12T18:49:17.948Z,2026-04-12T18:49:17.948Z,/Repos/ravi.ara@northeastern.edu/FoodInspection_Chicago_Dallas/notebooks/04_gold_dim_load
2633654,16738,5322,51,20260408,Pass,90,Chicago,1,2026-04-12T18:49:17.948Z,2026-04-12T18:49:17.948Z,/Repos/ravi.ara@northeastern.edu/FoodInspection_Chicago_Dallas/notebooks/04_gold_dim_load
2633653,23963,13785,55,20260408,Pass,90,Chicago,2,2026-04-12T18:49:17.948Z,2026-04-12T18:49:17.948Z,/Repos/ravi.ara@northeastern.edu/FoodInspection_Chicago_Dallas/notebooks/04_gold_dim_load
2633638,13905,16740,43,20260408,Fail,70,Chicago,3,2026-04-12T18:49:17.948Z,2026-04-12T18:49:17.948Z,/Repos/ravi.ara@northeastern.edu/FoodInspection_Chicago_Dallas/notebooks/04_gold_dim_load
2633566,35307,14565,2,20260407,Pass,90,Chicago,4,2026-04-12T18:49:17.948Z,2026-04-12T18:49:17.948Z,/Repos/ravi.ara@northeastern.edu/FoodInspection_Chicago_Dallas/notebooks/04_gold_dim_load
2633540,18427,7634,57,20260407,Pass,90,Chicago,5,2026-04-12T18:49:17.948Z,2026-04-12T18:49:17.948Z,/Repos/ravi.ara@northeastern.edu/FoodInspection_Chicago_Dallas/notebooks/04_gold_dim_load
2633624,20457,17483,12,20260407,Fail,70,Chicago,6,2026-04-12T18:49:17.948Z,2026-04-12T18:49:17.948Z,/Repos/ravi.ara@northeastern.edu/FoodInspection_Chicago_Dallas/notebooks/04_gold_dim_load
2633586,3843,16003,12,20260407,Fail,70,Chicago,7,2026-04-12T18:49:17.948Z,2026-04-12T18:49:17.948Z,/Repos/ravi.ara@northeastern.edu/FoodInspection_Chicago_Dallas/notebooks/04_gold_dim_load
2633604,2896,14566,18,20260407,Pass,90,Chicago,8,2026-04-12T18:49:17.948Z,2026-04-12T18:49:17.948Z,/Repos/ravi.ara@northeastern.edu/FoodInspection_Chicago_Dallas/notebooks/04_gold_dim_load
2633595,29308,14567,2,20260407,Pass,90,Chicago,9,2026-04-12T18:49:17.948Z,2026-04-12T18:49:17.948Z,/Repos/ravi.ara@northeastern.edu/FoodInspection_Chicago_Dallas/notebooks/04_gold_dim_load


---
## 8. bridge_inspection_violation

In [0]:
# Reload fact and violation dim
fact_inspection = spark.table(f"{DATABASE_NAME}.fact_inspection")
dim_violation = spark.table(f"{DATABASE_NAME}.dim_violation")

In [0]:
# Chicago bridge
df_chi_bridge = (
    df_chi_viol
    .join(
        fact_inspection.select("inspection_key", "inspection_id", "source_city"),
        (df_chi_viol["Inspection_ID"] == fact_inspection["inspection_id"]) &
        (df_chi_viol["source_city"] == fact_inspection["source_city"]),
        "inner"
    )
    .join(
        dim_violation,
        (df_chi_viol["violation_code"] == dim_violation["violation_code"]) &
        (df_chi_viol["source_city"] == dim_violation["source_city"]),
        "inner"
    )
    .select(
        fact_inspection["inspection_key"],
        dim_violation["violation_key"],
        df_chi_viol["violation_comments"]
    )
)

# Dallas bridge
df_dal_bridge = (
    df_dal_viol
    .join(
        fact_inspection.select("inspection_key", "inspection_id", "source_city"),
        (df_dal_viol["Inspection_ID"] == fact_inspection["inspection_id"]) &
        (df_dal_viol["source_city"] == fact_inspection["source_city"]),
        "inner"
    )
    .join(
        dim_violation,
        (df_dal_viol["violation_code"] == dim_violation["violation_code"]) &
        (df_dal_viol["source_city"] == dim_violation["source_city"]),
        "inner"
    )
    .select(
        fact_inspection["inspection_key"],
        dim_violation["violation_key"],
        df_dal_viol["violation_comments"]
    )
)

# Union and write
df_bridge = df_chi_bridge.unionByName(df_dal_bridge)

df_bridge = add_gold_lineage(df_bridge)

(
    df_bridge.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(f"{DATABASE_NAME}.bridge_inspection_violation")
)

print(f"bridge_inspection_violation: {df_bridge.count()} rows")

bridge_inspection_violation: 19718403 rows


---
## 9. Verify Gold Layer

In [0]:
display(spark.sql(f"""
    SELECT 'dim_date' AS table_name, COUNT(*) AS row_count FROM {DATABASE_NAME}.dim_date
    UNION ALL SELECT 'dim_restaurant', COUNT(*) FROM {DATABASE_NAME}.dim_restaurant
    UNION ALL SELECT 'dim_location', COUNT(*) FROM {DATABASE_NAME}.dim_location
    UNION ALL SELECT 'dim_inspection_type', COUNT(*) FROM {DATABASE_NAME}.dim_inspection_type
    UNION ALL SELECT 'dim_violation', COUNT(*) FROM {DATABASE_NAME}.dim_violation
    UNION ALL SELECT 'fact_inspection', COUNT(*) FROM {DATABASE_NAME}.fact_inspection
    UNION ALL SELECT 'bridge_inspection_violation', COUNT(*) FROM {DATABASE_NAME}.bridge_inspection_violation
"""))

table_name,row_count
dim_date,5939
dim_restaurant,45552
dim_location,34148
dim_inspection_type,64
dim_violation,990
fact_inspection,357094
bridge_inspection_violation,19718403
